In [ ]:
import os, re
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
raw_path = "./data/backpack/backpack_512x512x373_uint16.raw"
shape = (373, 512, 512)    # (Z, Y, X) = (slices, height, width)
voxel_spacing = (1.25, 0.9766, 0.9766)  # (z, y, x) mm，可用于3D可视化时设置比例
dtype = np.uint16

# dataload
vol = np.memmap(raw_path, dtype=dtype, mode="r", shape=shape)

In [ ]:
category  = 'backpack'
save_root = os.path.join(os.getcwd(), 'data_evaluate', category)

In [ ]:
# 简单校验：文件大小是否匹配
expected_bytes = np.prod(shape) * np.dtype(dtype).itemsize
actual_bytes = os.path.getsize(raw_path)
print(f"Expected bytes: {expected_bytes:,}, Actual bytes: {actual_bytes:,}")

In [ ]:
print("Path       :", raw_path)
print("Shape      :", vol.shape, "(Z, Y, X)")
print("Dtype      :", vol.dtype)
print("Itemsize   :", vol.dtype.itemsize, "bytes")
print("NDIM       :", vol.ndim)

In [ ]:
vmin, vmax = int(vol.min()), int(vol.max())
p1, p50, p99 = np.percentile(vol, [1, 50, 99])
print("\n——grayscale intensity——")
print(f"Min/Max    : {vmin} / {vmax}")
print(f"P1/Median/P99: {p1:.1f} / {p50:.1f} / {p99:.1f}")

In [ ]:
# Normalize data
def normalize_volume_minmax(volume):
    """最小-最大归一化到 [0, 1] 范围"""
    vol_min = np.min(volume)
    vol_max = np.max(volume)
    print(f"raw data range: [{vol_min}, {vol_max}]")
    
    if vol_min != 0 or vol_max != 1:
        print("Normalize range to [0, 1]")
        volume_normalized = (volume.astype(np.float32) - vol_min) / (vol_max - vol_min + 1e-8)
    else:
        volume_normalized = volume.astype(np.float32)
    
    print(f"normalized range: [{volume_normalized.min():.6f}, {volume_normalized.max():.6f}]")
    return volume_normalized

vol_normalized = normalize_volume_minmax(vol)

vmin, vmax = int(vol_normalized.min()), int(vol_normalized.max())
p1, p50, p995 = np.percentile(vol_normalized, [1, 50, 99.5])
print("\n——grayscale intensity——")
print(f"Min/Max    : {vmin} / {vmax}")
print(f"P1/Median/P99.5: {p1:.1f} / {p50:.1f} / {p995:.1f}")

In [ ]:
sample = np.asarray(vol[:: max(1, shape[0]//50), ::8, ::8]).ravel()  # 稀疏采样
plt.figure(figsize=(5,3))
plt.hist(sample, bins=256)
plt.title("Value Histogram (raw data)")
plt.xlabel("intensity")
plt.ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
sample = np.asarray(vol[:: max(1, shape[0]//50), ::8, ::8]).ravel()  # 稀疏采样
sample_nonzero = sample[sample != 0]
plt.figure(figsize=(5,3))
plt.hist(sample_nonzero, bins=256)
plt.title("Value Histogram (nonzero)")
plt.xlabel("intensity")
plt.ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
low, high = np.percentile(vol, [1, 99.5])  # 你也可以调成 [5, 99] 或手动写死
print(f"Auto window range: [{low:.1f}, {high:.1f}]")

def show_slice(img2d, vmin=None, vmax=None, title=""):
    plt.figure(figsize=(5,5))
    plt.imshow(img2d, cmap="gray", vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.axis("off")
    plt.show()

z_mid = shape[0] // 2
y_mid = shape[1] // 2
x_mid = shape[2] // 2

# Axial(轴向，Z固定，看 XY 平面)
show_slice(vol[z_mid, :, :], vmin=low, vmax=high, title=f"(z={z_mid})")

# Coronal(冠状，Y固定，看 XZ 平面)
show_slice(vol[:, y_mid, :], vmin=low, vmax=high, title=f"(y={y_mid})")

# Sagittal(矢状，X固定，看 YZ 平面)
show_slice(vol[:, :, x_mid], vmin=low, vmax=high, title=f"(x={x_mid})")


In [ ]:
from skimage.measure import marching_cubes
import trimesh

iso = np.percentile(vol, 99)  # 你可以微调阈值
verts, faces, normals, _ = marching_cubes(vol.astype(np.float32), level=iso, spacing=voxel_spacing)
mesh = trimesh.Trimesh(verts, faces, vertex_normals=normals, process=False)
mesh.show()

In [ ]:
# 灰度累积投影

import os
import time
import numpy as np
from scipy.ndimage import rotate
from tqdm import tqdm
from matplotlib import pyplot as plt

def _pad_to_safe_cube(vol, cval=0.0):
    """
    将(Z,Y,X)体数据填充为边长L的立方体，以避免任意绕Y/Z旋转时被裁剪。
    取 L = ceil(max(Z,Y,X) * sqrt(3))，并将原体素居中放置。
    返回：padded_vol (L,L,L)
    """
    z, y, x = vol.shape
    L = int(np.ceil(max(z, y, x) * np.sqrt(3)))  # 保守上界
    pad_z_total = max(0, L - z)
    pad_y_total = max(0, L - y)
    pad_x_total = max(0, L - x)
    pad_width = (
        (pad_z_total // 2, pad_z_total - pad_z_total // 2),
        (pad_y_total // 2, pad_y_total - pad_y_total // 2),
        (pad_x_total // 2, pad_x_total - pad_x_total // 2),
    )
    padded = np.pad(vol, pad_width, mode="constant", constant_values=cval)
    return padded

def project_volume_orthographic(
    vol_normalized,
    azimuth_deg=0.0,        # 水平转角（绕 Z 轴），x->y 为正
    elevation_deg=30.0,     # 俯仰角（绕 Y 轴），x->z 为正
    spacing=(1.0, 1.0, 1.0),# 体素尺寸(dz, dy, dx)
    mode="sum",             # {"sum","mean","mip"}
    order=1,                # 插值阶数：0=邻近，1=线性，3=三次样条
    cval=0.0,               # 旋转边界填充值
    safe_same_size=True     # True: 先pad到安全立方体，并固定输出尺寸
):
    """
    返回 2D 投影(np.float32)。旋转顺序：先绕Y轴(elevation)，再绕Z轴(azimuth)；投影方向为旋转后体数据的+Z方向。
    """
    vol = vol_normalized.astype(np.float32, copy=False)
    if safe_same_size:
        vol = _pad_to_safe_cube(vol, cval=cval)

    dz, dy, dx = spacing


    
    # 1) 俯仰角（绕 X 轴，上下倾斜）保持不变
    vol_rot = rotate(vol, angle=elevation_deg, axes=(1, 0), reshape=False,
                 order=order, mode='constant', cval=cval)

    # 2) 左右扫动（绕 Y 轴）
    vol_rot = rotate(vol_rot, angle=azimuth_deg, axes=(0, 2), reshape=False,
                 order=order, mode='constant', cval=cval)


    # 3) 沿 Z 轴做投影
    if mode == "sum":
        proj = vol_rot.sum(axis=0) * dz
    elif mode == "mean":
        proj = vol_rot.mean(axis=0)
    elif mode == "mip":
        proj = vol_rot.max(axis=0)
    else:
        raise ValueError("mode must be one of {'sum','mean','mip'}")

    return proj.astype(np.float32, copy=False)

def save_rotation_projections(
    vol_normalized,
    save_root,
    elevation_deg=0.0,
    mode="sum",
    step_deg=3.0,           # 每帧角度增量
    total_deg=360.0,        # 总角度
    order=1,
    cval=0.0,
    spacing=(1.0,1.0,1.0),
    progress_desc="Saving projections"
):
    """
    绕 Y 轴生成投影，输出目录和文件名格式：
    - 目录: Proj_elevation_{elevation_deg}
    - 文件: angle_{角度}.png
    """
    assert step_deg > 0, "step_deg 必须为正数"
    n_frames = int(np.ceil(total_deg / step_deg))
    azimuths = (np.arange(n_frames) * step_deg) % 360.0

    # 输出目录
    out_dir = os.path.join(save_root, f"Proj_elevation_{elevation_deg}")
    os.makedirs(out_dir, exist_ok=True)
    print(f"Save dir: {out_dir}")

    t0 = time.time()
    saved = []

    with tqdm(total=n_frames, desc=progress_desc) as pbar:
        for i, az in enumerate(azimuths):
            proj = project_volume_orthographic(
                vol_normalized,
                azimuth_deg=float(az),
                elevation_deg=float(elevation_deg),
                spacing=spacing,
                mode=mode,
                order=order,
                cval=cval,
                safe_same_size=True
            )
            # 文件名改为 angle_x.png
            fname = os.path.join(out_dir, f"angle_{int(round(az))}.png")
            plt.imsave(fname, proj, cmap="gray")
            saved.append((float(az), fname))
            pbar.set_postfix({"angle_deg": f"{az:.1f}"})
            pbar.update(1)

    elapsed = time.time() - t0
    print(f"⏱️ Total time: {elapsed:.2f} s | Frames: {n_frames} | Output dir: {out_dir}")

    return {
        "out_dir": out_dir,
        "frames": saved,
        "elapsed_sec": elapsed,
        "n_frames": n_frames
    }


In [ ]:
info = save_rotation_projections(
    vol,
    save_root=save_root,
    elevation_deg=30.0,
    mode="sum",
    step_deg=3.0,
    total_deg=360.0,
    order=1,
    cval=0.0,
    spacing=(1.0,1.0,1.0),
    progress_desc="Projection loop"
)

In [ ]:
# # 渲染后投影
# from tqdm import tqdm
# from skimage import measure
# import matplotlib
# from matplotlib import pyplot as plt
# from mpl_toolkits.mplot3d.art3d import Poly3DCollection
# import imageio.v2 as iio



# # ---- 参数设置（可改）
# proj_num       = 60                                # 渲染视角数量
# angle_interval = 360.0 / proj_num
# elevation      = 30                                # 仰角 [elevation]
# dx             = 3                                 # 视角小偏移（保持你原来的风格）
# sigma          = 0.6                               # 阈值混合系数
# alpha_mesh     = 0.30                              # 网格透明度（face alpha）
# face_color     = [0.5, 0.5, 0.5]                   # 网格颜色 (灰)
# dpi_save       = 200                               # 导出图片 DPI
# make_gif       = True                              # 是否合成 GIF
# gif_fps        = 24                                # GIF 帧率


# # ---- 阈值（sigma*min + (1-sigma)*max）
# vmin, vmax = float(vol_normalized.min()), float(vol_normalized.max())
# threshold  = sigma * vmin + (1.0 - sigma) * vmax   # 在[0,1]下等价 ~ 1 - (1-sigma)
# print(f"[Info] marching_cubes level (threshold): {threshold:.4f}  (min={vmin:.4f}, max={vmax:.4f})")

# verts, faces, normals, _ = measure.marching_cubes(vol_normalized, level=threshold)

# # ---- 组织输出目录
# series_save_dir = os.path.join(
#     save_root, f"elevation_{elevation}_sigma_{sigma}_alpha_{alpha_mesh}/"
# )
# os.makedirs(series_save_dir, exist_ok=True)

# # ---- 画布与三维坐标轴
# fig = plt.figure(figsize=(10, 10))
# ax  = fig.add_subplot(111, projection='3d')

# # 构建三角面集合
# mesh = Poly3DCollection(verts[faces], alpha=alpha_mesh, edgecolor='none')
# mesh.set_facecolor(face_color)
# ax.add_collection3d(mesh)

# # 轴范围（与原代码逻辑相同）
# Z, Y, X = vol_normalized.shape
# ax.set_xlim(0, Z)
# ax.set_ylim(0, Y)
# ax.set_zlim(0, X)
# ax.axis("off")

# # ---- 批量渲染并保存
# img_files = []
# for i in tqdm(range(proj_num), desc="Rendering views"):
#     angle = angle_interval * i + dx
#     ax.view_init(elev=elevation, azim=angle)
#     # 紧边距 & 关闭边框
#     plt.tight_layout(pad=0)
#     out_path = os.path.join(series_save_dir, f'angle_{int(round(angle))}.png')
#     plt.savefig(out_path, dpi=dpi_save)
#     img_files.append(out_path)
    
# plt.close(fig)  # 释放内存
# print(f"[Done] Saved {len(img_files)} views to: {series_save_dir}")

# # ---- 可选：合成 GIF
# if make_gif and len(img_files) > 0:
#     gif_name = os.path.join(
#         series_save_dir, f'rotate_{category}_fps_{gif_fps}.gif'
#     )
#     imgs = [iio.imread(p) for p in img_files]
#     # duration 按帧率换算（秒/帧）
#     iio.mimsave(gif_name, imgs, duration=1.0/gif_fps, loop=0)
#     print(f"[Done] GIF saved to: {gif_name}")